# ChurnGuard AI - Model Training
Loads the train/test splits produced by `data_prep.ipynb`, trains three
candidate models (Random Forest, XGBoost, LightGBM), evaluates them with
ROC-AUC and PR-AUC (accuracy is not reliable here given the ~26.5% churn
rate), logs every run to MLflow, and registers the best model in the
MLflow Model Registry so the API can load it by name later.

Class imbalance is handled with `class_weight="balanced"` (Random Forest)
or `scale_pos_weight` (XGBoost, LightGBM), computed from the training
set only.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

DATA_DIR = "../data/processed"
MODELS_DIR = "../models"
RANDOM_STATE = 42

os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Load train/test splits
Output of `data_prep.ipynb`. `X_train`/`X_test` are already encoded and
column-aligned; `feature_columns.joblib` holds the exact column order the
model expects, which the API will reuse.

In [ ]:
X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv")).squeeze()
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv")).squeeze()

feature_columns = joblib.load(os.path.join(MODELS_DIR, "feature_columns.joblib"))
X_train = X_train[feature_columns]
X_test = X_test[feature_columns]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.1%}, Test churn rate: {y_test.mean():.1%}")

## 2. MLflow setup
Local tracking server, file-based backend store. Run `mlflow ui` in a
separate terminal from the project root to view the dashboard at
`http://127.0.0.1:5000`.

In [ ]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("churnguard-model-selection")

# Imbalance ratio, used by XGBoost/LightGBM's scale_pos_weight
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"Negative: {neg}, Positive: {pos}, scale_pos_weight: {scale_pos_weight:.2f}")

## 3. Shared evaluation helper
Every model is scored the same way and logged to MLflow the same way, so
runs are directly comparable in the dashboard. PR-AUC is emphasized
alongside ROC-AUC since it is more informative under class imbalance.

In [ ]:
def evaluate_and_log(model, model_name, params, log_fn):
    """Fit inside an MLflow run, log params/metrics/artifacts, return the fitted model."""
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)
        f1 = f1_score(y_test, y_pred)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

        mlflow.log_params(params)
        mlflow.log_metrics({
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "f1_score": f1,
            "cv_roc_auc_mean": cv_scores.mean(),
            "cv_roc_auc_std": cv_scores.std(),
        })

        cm = confusion_matrix(y_test, y_pred)
        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(ax=ax, colorbar=False)
        ax.set_title(model_name)
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        log_fn(model)

        print(f"{model_name}: ROC-AUC={roc_auc:.4f}  PR-AUC={pr_auc:.4f}  "
              f"F1={f1:.4f}  CV ROC-AUC={cv_scores.mean():.4f} (+/-{cv_scores.std():.4f})")
        print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

        return model, run.info.run_id, {"roc_auc": roc_auc, "pr_auc": pr_auc, "f1_score": f1}

## 4. Random Forest (baseline)
`class_weight="balanced"` reweights classes inversely to their frequency,
no resampling needed.

In [ ]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 10,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}
rf_model = RandomForestClassifier(**rf_params)

rf_model, rf_run_id, rf_metrics = evaluate_and_log(
    rf_model,
    "random_forest",
    rf_params,
    log_fn=lambda m: mlflow.sklearn.log_model(m, "model"),
)

## 5. XGBoost
`scale_pos_weight` handles the imbalance directly; `max_depth` and
`min_child_weight` are kept conservative given the small dataset size
(~5600 training rows) to limit overfitting.

In [ ]:
xgb_params = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "random_state": RANDOM_STATE,
    "eval_metric": "logloss",
}
xgb_model = XGBClassifier(**xgb_params)

xgb_model, xgb_run_id, xgb_metrics = evaluate_and_log(
    xgb_model,
    "xgboost",
    xgb_params,
    log_fn=lambda m: mlflow.xgboost.log_model(m, "model"),
)

## 6. LightGBM
`num_leaves` and `min_child_samples` are kept low on purpose: LightGBM's
leaf-wise growth can overfit fast on a dataset this small if left at
default settings.

In [ ]:
lgbm_params = {
    "n_estimators": 300,
    "num_leaves": 20,
    "max_depth": 5,
    "learning_rate": 0.05,
    "min_child_samples": 30,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "random_state": RANDOM_STATE,
    "verbose": -1,
}
lgbm_model = LGBMClassifier(**lgbm_params)

lgbm_model, lgbm_run_id, lgbm_metrics = evaluate_and_log(
    lgbm_model,
    "lightgbm",
    lgbm_params,
    log_fn=lambda m: mlflow.lightgbm.log_model(m, "model"),
)

## 7. Compare runs and select the best model
Selection is based on test PR-AUC (more informative than ROC-AUC under
class imbalance), with ROC-AUC as a tie-breaker.

In [ ]:
results = pd.DataFrame([
    {"model": "random_forest", "run_id": rf_run_id, **rf_metrics},
    {"model": "xgboost", "run_id": xgb_run_id, **xgb_metrics},
    {"model": "lightgbm", "run_id": lgbm_run_id, **lgbm_metrics},
]).sort_values("pr_auc", ascending=False).reset_index(drop=True)

results

In [ ]:
best = results.iloc[0]
print(f"Best model: {best['model']}  (PR-AUC={best['pr_auc']:.4f}, ROC-AUC={best['roc_auc']:.4f})")
print(f"Run ID: {best['run_id']}")

## 8. Register the best model in the MLflow Model Registry
Registering makes the model loadable by name/stage rather than by run ID.
The API will later load it with:
`mlflow.pyfunc.load_model("models:/churnguard-classifier/Production")`.

The model is registered here and promoted to the `Staging` stage; manual
promotion to `Production` is left as a deliberate checkpoint rather than
automated, so a human reviews the comparison table above first.

In [ ]:
model_uri = f"runs:/{best['run_id']}/model"
registered_name = "churnguard-classifier"

registered_model = mlflow.register_model(model_uri=model_uri, name=registered_name)

client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name=registered_name,
    version=registered_model.version,
    stage="Staging",
)

print(f"Registered '{registered_name}' version {registered_model.version} -> stage: Staging")
print("Promote to Production manually in the MLflow UI once validated, "
      "or with client.transition_model_version_stage(..., stage='Production').")

## 9. Save a local copy for quick reuse (optional)
Convenient for local scripts/tests that don't want to depend on the
MLflow tracking server being up. The registry (step 8) remains the
source of truth the API should use.

In [ ]:
best_model_obj = {"random_forest": rf_model, "xgboost": xgb_model, "lightgbm": lgbm_model}[best["model"]]
joblib.dump(best_model_obj, os.path.join(MODELS_DIR, "best_model.joblib"))
print(f"Saved local copy of '{best['model']}' to {os.path.join(MODELS_DIR, 'best_model.joblib')}")

## Summary
- Three models trained and logged to MLflow: Random Forest, XGBoost,
  LightGBM, each with the same evaluation protocol for a fair comparison.
- Class imbalance (~26.5% churn) handled via `class_weight="balanced"`
  (Random Forest) and `scale_pos_weight` (XGBoost, LightGBM), computed
  from the training set only.
- Evaluation uses ROC-AUC and PR-AUC rather than accuracy, plus 5-fold
  stratified cross-validation on ROC-AUC to sanity-check stability.
- Model selection is based on test PR-AUC, the more informative metric
  under class imbalance.
- The winning model is registered in the MLflow Model Registry under
  `churnguard-classifier` and promoted to the `Staging` stage; promotion
  to `Production` is a manual, deliberate step.
- Next step: build the FastAPI service that loads the model from the
  registry (`models:/churnguard-classifier/Production`) together with the
  saved encoder and feature column order from `data_prep.ipynb`, to score
  new customers exactly the same way as during training.